# LiveCodeBench

This example shows how to evaluate a `genlm.control` model on the LiveCodeBench domain.

* **Task**: Generate a correct Python program for a competitive-programming problem (stdin/stdout or a function to implement).
* **Data**: LiveCodeBench `code_generation_lite` (Jain et al., 2024).

## Setup

First, install the dependencies for this domain. In the root directory, run:

```bash
pip install -e .[livecodebench] genlm-control
```

## Usage

### Initialize the dataset and evaluator

`from_hf` downloads and decodes a release of `code_generation_lite`. `cumulative=True` gives the official `version_tag` semantics (all problems through that release); `start_date` restricts the contest window (the default `2024-01-01` is after the Llama-3.x cutoffs). The first call needs network access to populate the Hugging Face cache.

In [ ]:
from genlm.eval.domains.livecodebench import (
    LiveCodeBenchDataset, LiveCodeBenchEvaluator
)

dataset = LiveCodeBenchDataset.from_hf(
    release="release_v6",
    start_date="2024-01-01",
    max_instances=8,
)

print("Instances loaded:", len(dataset))
evaluator = LiveCodeBenchEvaluator(timeout_seconds=6.0)

### Inspect dataset

In [ ]:
first = next(iter(dataset))
print("Question ID:", first.instance_id)
print("Difficulty:", first.difficulty)
print("Test type:", first.testtype)
print("Prompt preview:\n", (first.question_content[:500] + "...") if len(first.question_content) > 500 else first.question_content)

## Model Adaptor

`default_prompt_formatter` builds the official `lcb_runner` prompt (chat template for instruct models); we sample unconstrained and apply `DEFAULT_STOP` (lcb_runner's `--stop "###"`). This domain ships no constraint potential.

In [ ]:
from genlm.control import PromptedLLM, direct_token_sampler
from genlm.eval import ModelOutput, ModelResponse
from genlm.eval.domains.livecodebench import DEFAULT_STOP, default_prompt_formatter

# Load an instruct LLM (chat template applied automatically below).
LLM = PromptedLLM.from_name("meta-llama/Llama-3.1-8B-Instruct", temperature=0.2)


async def model(instance, output_dir, replicate):
    # Build the official LCB prompt for this instance.
    LLM.prompt_ids = default_prompt_formatter(
        LLM.model.tokenizer, instance, use_chat_format=True
    )

    # Unconstrained sampling from the LLM (no constraint potential).
    sampler = direct_token_sampler(LLM)
    sequences = await sampler.smc(
        n_particles=5,
        ess_threshold=0.5,
        max_tokens=512,
    )

    def truncate(text):
        # Truncate at the official lcb_runner stop sequence (vLLM --stop "###").
        for stop in DEFAULT_STOP:
            text = text.split(stop)[0]
        return text

    return ModelOutput(
        responses=[
            ModelResponse(response=truncate(sequence), weight=prob)
            for sequence, prob in sequences.decoded_posterior.items()
        ],
    )

### Run the evaluation

In [ ]:
from genlm.eval import run_evaluation

results = await run_evaluation(
    dataset=dataset,
    model=model,
    evaluator=evaluator,
    max_instances=2,
    n_replicates=1,
    verbosity=1,
    #output_dir="livecodebench_results", #optionally save the results to a directory
)

Naman Jain, King Han, Alex Gu, Wen-Ding Li, Fanjia Yan, Tianjun Zhang, Sida Wang, Armando Solar-Lezama, Koushik Sen, and Ion Stoica. LiveCodeBench: Holistic and contamination free evaluation of large language models for code. arXiv preprint arXiv:2403.07974, 2024. URL https://arxiv.org/abs/2403.07974